# VisionFind GPU backend

Use a Kaggle or Colab CUDA GPU session with internet enabled.
This notebook starts the packaged backend without development patch cells.

First run installs dependencies and downloads model weights.
Restart the session after setup if previously imported packages changed.
The CUDA PyTorch/Torchvision pair must already be compatible.

Fresh model startup of this packaged notebook remains unverified.


In [ ]:
from pathlib import Path
import subprocess
import sys

base = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
project = base / "VisionFind"

if not project.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/Ibrahimshah0900/VisionFind.git",
         str(project)],
        check=True,
    )
assert (project / "run_backend.py").is_file(), "Invalid project checkout."

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-r", str(project / "requirements-backend.txt")],
    check=True,
)
print("Dependencies installed. Restart the Python session if needed.")
print("Keep the existing compatible CUDA torch/torchvision installation.")


## Start the backend

Enter a private random API token through the hidden prompt.
The server runs in a background thread while this session remains active.
Do not publish credentials in notebook outputs.


In [ ]:
from pathlib import Path
import os
import sys
import shutil
import getpass
import importlib.util

base = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
project = base / "VisionFind"
assert (project / "run_backend.py").is_file(), "Run setup first."
assert shutil.which("ffmpeg"), "FFmpeg is required."

os.environ["VISIONFIND_MODEL_CACHE"] = str(base / "VisionFind_models")
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

spec = importlib.util.spec_from_file_location(
    "visionfind_backend_launcher", project / "run_backend.py"
)
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)

# Enter a random token of at least 32 characters.
# Use the same token when configuring the desktop interface.
api_token = getpass.getpass("Backend API token — hidden: ").strip()
assert len(api_token) >= 32, "Token must contain at least 32 characters."

engine = launcher.build_engine()

from src.multi_chat import build_multi_chat
from src.api_server import create_api
import uvicorn
import threading
import time
import httpx

assert not (
    "api_thread" in globals() and api_thread.is_alive()
), "An API is already running; stop it before rerunning."

chat = build_multi_chat(engine)
api = create_api(chat.visionfind_chat_handler, project, api_token)
server = uvicorn.Server(
    uvicorn.Config(api, host="127.0.0.1", port=8000, log_level="warning")
)
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

for _ in range(100):
    try:
        response = httpx.get(
            "http://127.0.0.1:8000/health",
            headers={"Authorization": "Bearer " + api_token},
            timeout=2,
        )
        if response.status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(0.2)
else:
    raise RuntimeError("Backend health check did not succeed.")

print("Authenticated backend is running on localhost:8000.")
print("A separate HTTPS tunnel is needed for the desktop connection.")


## Desktop connection

Expose localhost:8000 through your HTTPS tunnel, then run
frontend/configure.py on your desktop with the tunnel URL and the same token.

The tunnel is separate from this notebook. The desktop frontend and GPU
runtime must remain running. Video explanations retain known accuracy limits.
